# Evaluation Pipeline: Scoring Extracted Variables against the Ground Truth (final, version 2)

## Purpose

*Version 2 (24 October 2026) differs from `evaluation_pipeline_final.ipynb` only in the exclusion of withdrawn reports (`EXCLUDED_REPORTS`, `apply_exclusions` and the Safeguards entry below) and of superseded prompt versions (`EXCLUDED_PROMPT_VERSIONS`: FS1_v1 and FS2_v1, which used worked examples that were later corrected).*

This notebook scores the model outputs recorded in the audit log of the prompting pipeline (`prompting_pipeline_final.ipynb`) against the annotated ground truth in `report_dataset_final_v10.xlsx`, and measures how consistent the repeated runs of the same prompt are. It reads the 28-variable schema, the empty-value standard and the metrics agreed at the supervisor meeting of 17 October 2026, and the agreement metrics decided on 21 October 2026.

This version differs from the earlier evaluation notebooks (v3 and v4) in three ways: (1) agreement between repeats is computed (Section 4); (2) the Excel workbook carries no colour formatting; (3) the demonstration run and the evaluation of a study stage are two separate functions, each run from its own cell (Sections 11 and 12), in the same way as the prompting pipeline.

## Scoring design

| Variable group | Variables | Score per variable and report | Metrics reported |
|---|---|---|---|
| **Boolean flags** (8) | the `*_flag` variables | exact match, with a blank (null) treated as a valid class | exact match; confusion counts and blank rates |
| **Structured variables** (6) | `file_number`, `sex`, `age_at_death`, `country_of_birth`, `date_of_findings`, `date_of_death` | exact match after normalisation (case, white space) | exact match |
| **Text variables** (14) | all remaining variables, including the six list variables | BERTScore F1 (the primary score) | BERTScore F1 (raw and rescaled with the baseline), ROUGE-2 F1, ROUGE-L F1 and SacreBLEU sentence BLEU (each scaled to 0 to 1) |

- **Empty values.** `None`, an empty string, an empty list and a string containing only white space are all treated as one "empty" value on both sides. Two empty values score 1.0 on every metric, and an empty value compared with a non-empty value scores 0.0 on every metric.
- **Spelling variants.** The reports use both "methamphetamine" and "methylamphetamine". Both spellings are replaced by "methamphetamine" on both sides before scoring (`SPELLING_VARIANTS`, applied in `to_text`), so a model is not penalised for the spelling that the report used.
- **`date_of_death` is scored by exact match** (decided 21 October 2026). The answers are short, even when the report gives a qualified date (for example "on or before ..." or a range of years), and BERTScore barely separates dates that differ by one digit (for example 8 August 2020 and 9 August 2020), which should be penalised fully. The text must therefore match the ground truth after normalisation.
- **List variables** are joined into one string with a comma and a space (a semicolon in the ground truth is read as a comma), and are then scored as text.
- **Identical strings.** After normalisation, identical strings score 1.0 on every metric. This avoids the artefact that ROUGE-2 and BLEU are undefined or zero for very short strings.
- **Prompt versions.** The prompting pipeline records every change to the wording of a prompt as a new version (`Z4_v1`, `Z4_v2`, and so on). Each version is scored as a separate prompt, so that results obtained with different wordings are never mixed. Logs from earlier pipeline versions have no version and keep the plain prompt name.
- **Best-prompt rule** (fixed on 17 October, before any validation result was observed). For each prompt version, the score of every variable is averaged over reports and repeats; the unweighted mean of the 28 variable scores is the prompt score; the highest score ranks first; and ties go to the shorter prompt (Section 8).

## Agreement between repeats

Agreement measures how consistent the repeated runs of one prompt on one report are. It is reported separately from accuracy: a prompt can be consistently wrong, and a correct answer can be unstable. Agreement is computed for every prompt version that has at least two repeats of a report (prompt development and internal validation: 5 repeats; external validation: 1 repeat, or 3 if the optional repeats are run).

| Variables | Agreement measure |
|---|---|
| All 28 variables | percentage of identical answers: the share of pairs of repeats that give the same normalised answer (case-insensitive) |
| Flags (8) and structured variables (6) | Krippendorff's alpha for nominal data (the repeats are the raters and the report-variable combinations are the items), pooled over the variables within each prompt version |
| Text variables (14) | mean pairwise similarity between the repeats, using BERTScore F1 (10 pairs for 5 repeats), with the percentage of identical answers alongside |
| List variables (6) | as for the text variables, and in addition the Jaccard similarity of the items in the lists |

## Safeguards

- **Prompt development only.** Only the stages in `ALLOWED_STAGES` can be scored, and the ground truth is read from the worksheet of that stage only. The internal and external validation datasets are therefore not read at this point of the study.
- **Demonstration run.** `run_demo` scores the records of the test report (the smoke-test record, or the 5 repeats of the demonstration run) against a **demonstration ground truth** that is derived from the records themselves with deliberate errors. It only checks that the notebook works; its numbers carry no information about extraction quality, and its outputs are written to a separate, labelled file.
- **Test records.** Records written in test mode (`test_mode = true`) are excluded from every study stage.
- **Withdrawn report.** Report 15 was withdrawn from the Coroners Court website, so the model received an error page instead of the report and its 40 trials are empty. The reports listed in `EXCLUDED_REPORTS` (Section 1) are removed from the trials and the ground truth before anything is scored, and a warning is printed if the prompt versions were not all run on the same reports. The replacement report is added to the ground truth worksheet after it has been annotated.

## Requirements

Python 3.10 or later and the packages `pandas`, `numpy`, `openpyxl`, `sacrebleu`, `rouge-score` and `bert-score` (the last requires `torch` and downloads `roberta-large` on first use).

## 1. Setup and settings

In [1]:
from pathlib import Path
from datetime import datetime
from itertools import combinations
import json, re, importlib.metadata as md_
import numpy as np
import pandas as pd
import sacrebleu
from bert_score import score as bert_score
from openpyxl.styles import Alignment, Font
from rouge_score import rouge_scorer

# Files and folders
WORKBOOK = "report_dataset_final_v10.xlsx"
RESULTS_ROOT = Path("results")     # audit logs of the study stages (the prompting pipeline writes results_test/ in test mode)
EVAL_ROOT = Path("evaluation")     # the evaluation outputs are written here, one sub-folder per stage (and DEMO)
DEMO_LOGS = [Path("results_test/test_report/audit_log.jsonl"), Path("results_test/smoke_test/audit_log.jsonl")]   # test-report records

# Only these sets can be scored. Widen the set deliberately and in order: internal_validation after the best prompt of each technique has been chosen, external_validation after the single best prompt has been chosen.
ALLOWED_STAGES = {"prompt_development"}

# Reports that are never scored, with the reason. Report 15 was withdrawn from the Coroners Court website on or before 24 October 2026, so the model received an error page instead of the report and all 40 of its trials are empty.
# Its trials stay in the audit log; they are removed here, together with its ground truth, before anything is scored.
# Report 51 (file 2021/00346986, draw 23 of the seed 2804 sequence) replaces it in the prompt development set.
EXCLUDED_REPORTS = {15: "withdrawn from the Coroners Court website (24 October 2026); the model did not receive the report"}

# Prompt versions that are never scored, with the reason. FS1_v1 and FS2_v1 were run with worked examples (EX1, EX2) whose annotations contained mismatches with the report PDFs; the examples were corrected on 24 October 2026 and both prompts were run again as FS1_v2 and FS2_v2. The first-run trials stay in the audit log; they are removed before anything is scored.
EXCLUDED_PROMPT_VERSIONS = {
    "FS1_v1": "worked examples contained annotation mismatches (corrected 24 October 2026); superseded by FS1_v2",
    "FS2_v1": "worked examples contained annotation mismatches (corrected 24 October 2026); superseded by FS2_v2",
}

# Spelling variants that are treated as the same word on both sides (ground truth and model output) before scoring. The reports
# use both "methamphetamine" and "methylamphetamine" (and "Methyl-Amphetamine"); an answer should not lose points for the
# spelling that the report happened to use.
SPELLING_VARIANTS = [(r"methyl-?amphetamine", "methamphetamine")]

# Scoring settings
PRIMARY_TEXT_METRIC = "bert_f1_rescaled"   # "bert_f1_rescaled" (baseline-rescaled, clipped to 0..1) or "bert_f1_raw"
BERT_MODEL, BERT_LANG, BERT_BATCH = "roberta-large", "en", 16
TIE_DECIMALS = 4                           # prompt scores are rounded to this many decimals before ties are resolved
print(f"Stages that can be scored: {sorted(ALLOWED_STAGES)} | primary text metric: {PRIMARY_TEXT_METRIC}")

/opt/anaconda3/envs/eval-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Stages that can be scored: ['prompt_development'] | primary text metric: bert_f1_rescaled


## 2. Variable groups

The groups below follow the scoring design in the introduction. The assertion at the end of the cell checks that the groups together contain exactly the 28 variables of the ground-truth sheet, so that a variable added to or removed from the schema cannot go unscored. Only the header row of the sheet `prompt_development` is read here. `date_of_death` is in the structured group: it is scored by exact match because BERTScore would hardly penalise a date that is wrong by one day.

In [2]:
FLAG_FIELDS = ["intellectual_disability_flag", "mental_health_flag", "substance_use_flag", "custody_served_flag",
               "non_english_speaker_flag", "use_of_interpreters_flag", "preventable_death_flag", "domestic_violence_flag"]
EXACT_FIELDS = ["file_number", "sex", "age_at_death", "country_of_birth", "date_of_findings", "date_of_death"]   # date_of_death: short phrases, so exact match (changed 21 Oct)
LIST_FIELDS = ["potential_causes_of_death", "medications", "intellectual_disability_conditions", "mental_health_conditions",
               "substances_used", "known_health_issues"]                      # list[str] in the output schema; scored as text
TEXT_FIELDS = ["place_of_death", "definitive_cause_of_death", "manner_of_death", "place_of_residence",
               "profession", "level_of_education", "toxicology_results_summary", "coroners_recommendations_summary"] + LIST_FIELDS

FIELD_TYPE = {**{f: "flag" for f in FLAG_FIELDS}, **{f: "structured" for f in EXACT_FIELDS}, **{f: "text" for f in TEXT_FIELDS}}
_gt_columns = [c for c in pd.read_excel(WORKBOOK, sheet_name="prompt_development", nrows=0).columns if c not in ("Report No.", "Random date selection")]
FIELDS = _gt_columns                                                           # the order of the ground-truth sheet
assert set(FIELD_TYPE) == set(FIELDS) and len(FIELDS) == 28, set(FIELD_TYPE) ^ set(FIELDS)
print({t: sum(v == t for v in FIELD_TYPE.values()) for t in ("flag", "structured", "text")})

{'flag': 8, 'structured': 6, 'text': 14}


## 3. Normalisation and scoring metrics

`to_text` converts any value (null, string, number, boolean or list) to a normalised string, so that the two sides of every comparison are in the same form. The lexical metrics (ROUGE-2, ROUGE-L, SacreBLEU) are computed by `lexical_metrics`. BERTScore is computed in one batch over all distinct text pairs by `bertscore_batch`, because the pairs repeat heavily across repeats and prompts; both the raw F1 and the F1 rescaled with the baseline of the model are stored. BERTScore truncates each string at 512 tokens, which affects only the longest narrative variables (toxicology and recommendations).

In [3]:
_ROUGE = rouge_scorer.RougeScorer(["rouge2", "rougeL"], use_stemmer=True)
_WS = re.compile(r"\s+")


def unify_spelling(s: str) -> str:
    """Return the string with the spelling variants in SPELLING_VARIANTS replaced by one spelling (the capital letter is kept).

    Args:
        s: The text.

    Returns:
        The text with each variant replaced.
    """
    for pattern, standard in SPELLING_VARIANTS:
        s = re.sub(pattern, lambda m: standard.capitalize() if m.group(0)[0].isupper() else standard, s, flags=re.IGNORECASE)
    return s


def to_text(v, field: str | None = None) -> str:
    """Return the normalised string form of any value; the empty value is ''.

    Spelling variants listed in SPELLING_VARIANTS (for example methylamphetamine) are replaced by one spelling.
    Lists are joined with ', ', booleans become 'TRUE'/'FALSE', whole numbers lose their decimal part, and white space is collapsed. For list variables, ';' and ',' separators in a string are rewritten as ', '.

    Args:
        v: The value (None, NaN, list, bool, number or string).
        field: The variable name, used to recognise list variables.

    Returns:
        The normalised string.
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (list, tuple)):
        return unify_spelling(", ".join(s for s in (_WS.sub(" ", str(x)).strip() for x in v) if s))
    if isinstance(v, (bool, np.bool_)):
        return "TRUE" if v else "FALSE"
    if isinstance(v, (int, float, np.integer, np.floating)):
        return str(int(v)) if float(v).is_integer() else str(v)
    s = _WS.sub(" ", str(v)).strip()
    if field in LIST_FIELDS:
        s = re.sub(r"\s*[;,]\s*", ", ", s).strip(", ")                    # one separator convention for list variables
    return unify_spelling(s)


def to_flag(v) -> str:
    """Return a flag value as 'TRUE', 'FALSE' or '' (blank).

    Args:
        v: The flag value.

    Returns:
        'TRUE', 'FALSE' or ''.

    Raises:
        ValueError: If the value is anything other than TRUE, FALSE or blank.
    """
    t = to_text(v).upper()
    if t not in ("TRUE", "FALSE", ""):
        raise ValueError(f"A flag must be TRUE, FALSE or blank, found {v!r}.")
    return t


def lexical_metrics(pred: str, gt: str) -> dict:
    """Return the lexical similarity of a prediction to the ground truth.

    Args:
        pred: The model's normalised text.
        gt: The ground-truth normalised text.

    Returns:
        A dict with rouge2_f1, rougeL_f1 and bleu (SacreBLEU sentence BLEU scaled to 0..1).
    """
    r = _ROUGE.score(gt, pred)                                               # (target, prediction)
    return dict(rouge2_f1=r["rouge2"].fmeasure, rougeL_f1=r["rougeL"].fmeasure,
                bleu=sacrebleu.sentence_bleu(pred, [gt]).score / 100)


def bertscore_batch(pairs: list[tuple[str, str]]) -> dict:
    """Return the BERTScore F1 of distinct non-empty (pred, gt) pairs, in one batch.

    The BERTScore hashes of the raw and rescaled runs are stored in `bertscore_batch.hashes` for the README.

    Args:
        pairs: The (prediction, ground truth) pairs to score.

    Returns:
        A dict mapping each (pred, gt) pair to (raw_f1, rescaled_f1).
    """
    if not pairs:
        return {}
    cands, refs = [p for p, _ in pairs], [g for _, g in pairs]
    kw = dict(lang=BERT_LANG, model_type=BERT_MODEL, batch_size=BERT_BATCH, verbose=False)
    (_, _, raw), h_raw = bert_score(cands, refs, return_hash=True, **kw)
    (_, _, resc), h_resc = bert_score(cands, refs, rescale_with_baseline=True, return_hash=True, **kw)
    bertscore_batch.hashes = (h_raw, h_resc)
    return {pr: (float(a), float(b)) for pr, a, b in zip(pairs, raw.tolist(), resc.tolist())}
bertscore_batch.hashes = None

## 4. Agreement metrics between repeats

Each prompt version is run several times on each report. For one report and one variable, the repeats give a set of answers (for example five answers). The functions below measure how far these answers agree.

- **Percentage of identical answers** (`pairwise_identical`). All pairs of repeats are formed (10 pairs for 5 repeats) and the share of pairs with the same normalised answer (case-insensitive) is taken. A blank answer is a valid answer, so two blanks agree.
- **Krippendorff's alpha, nominal** (`krippendorff_alpha_nominal`). Alpha compares the observed disagreement between repeats with the disagreement expected by chance, given how often each answer occurs overall: alpha = 1 means perfect agreement, 0 means agreement no better than chance, and negative values mean systematic disagreement. Each report-variable combination is one item, and each repeat is one rater. A pooled alpha is computed over several variables within one prompt version. Alpha is **undefined** (not a number) when every answer in the pool is identical, because there is then no variation from which a chance level can be estimated. For a rare flag that is blank in almost every report, the percentage of identical answers is high, but alpha can be close to 0 or undefined; a low alpha for such a flag should not be over-interpreted.
- **Pairwise text similarity** (`pair_similarity`). For text variables, the similarity of two repeats is their BERTScore F1 (rescaled and clipped to 0 to 1 when `PRIMARY_TEXT_METRIC` is the rescaled score), with two identical answers scoring 1.0, two empty answers 1.0, and one empty answer 0.0, exactly as in the scoring against the ground truth. The agreement of a report and variable is the mean over all pairs of repeats. BERTScore F1 is symmetric, so the order of the two answers does not matter.
- **Jaccard similarity** (`jaccard`). For list variables (for example medications), each answer is split into its items (case-insensitive), and the similarity of two repeats is the number of items in common divided by the number of items in either (1.0 for identical lists, 0.0 for lists with nothing in common; two empty lists score 1.0 and one empty list 0.0). Items that contain a comma themselves are split at the comma, as the list is stored as comma-separated text.

Alpha is a measure of consistency and does not measure accuracy against the ground truth.

In [4]:
def pairwise_identical(answers: list[str]) -> float:
    """Return the share of pairs of answers that are identical (case-insensitive).

    Args:
        answers: The normalised answers of the repeats of one report and variable.

    Returns:
        The mean over all pairs, or NaN if there are fewer than two answers.
    """
    v = [a.casefold() for a in answers]
    pairs = list(combinations(v, 2))
    return float(np.mean([a == b for a, b in pairs])) if pairs else np.nan


def krippendorff_alpha_nominal(units: list[list[str]]) -> float:
    """Return Krippendorff's alpha for nominal data.

    Each unit is one item (one report and variable) and holds the answers of its repeats. Units with fewer than two answers are ignored. The coincidence matrix gives each pair of answers within a unit the weight 1/(m - 1), where m is the number of answers in the unit; alpha = 1 - D_observed / D_expected.

    Args:
        units: One list of answers (strings) per item.

    Returns:
        Alpha, or NaN when it is undefined (no pairable answers, or every answer in the pool is the same, so that no chance level can be estimated).
    """
    coincidence: dict[tuple[str, str], float] = {}
    for vals in units:
        m = len(vals)
        if m < 2:
            continue
        for i, c in enumerate(vals):
            for j, k in enumerate(vals):
                if i != j:
                    coincidence[(c, k)] = coincidence.get((c, k), 0.0) + 1.0 / (m - 1)
    totals: dict[str, float] = {}
    for (c, _), w in coincidence.items():
        totals[c] = totals.get(c, 0.0) + w
    n = sum(totals.values())
    if n <= 1:
        return np.nan
    observed = sum(w for (c, k), w in coincidence.items() if c != k)          # weight of disagreeing pairs
    expected = n * n - sum(v * v for v in totals.values())                     # sum over c != k of n_c * n_k
    if expected == 0:
        return np.nan
    return 1.0 - (n - 1) * observed / expected


def split_items(text: str) -> set[str]:
    """Return the set of items (lower case) of a list answer stored as comma-separated text.

    Args:
        text: The normalised answer.

    Returns:
        The set of non-empty items.
    """
    return {s.strip().casefold() for s in text.split(",") if s.strip()}


def jaccard(a: str, b: str) -> float:
    """Return the Jaccard similarity of the items of two list answers.

    Args:
        a: The first normalised answer.
        b: The second normalised answer.

    Returns:
        Items in common divided by items in either; 1.0 for two empty lists and 0.0 when exactly one list is empty.
    """
    sa, sb = split_items(a), split_items(b)
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb)


def pair_similarity(a: str, b: str, bs: dict) -> float:
    """Return the similarity of two text answers (BERTScore F1, 0 to 1).

    Two empty answers score 1.0, one empty answer 0.0, and two identical answers (case-insensitive) 1.0; any other pair is looked up in the BERTScore results.

    Args:
        a: The first normalised answer.
        b: The second normalised answer.
        bs: The BERTScore results, keyed by the sorted pair of answers.

    Returns:
        The similarity, clipped to the range 0 to 1.
    """
    if a == "" and b == "":
        return 1.0
    if a == "" or b == "":
        return 0.0
    if a.casefold() == b.casefold():
        return 1.0
    raw, rescaled = bs[tuple(sorted((a, b)))]
    return float(min(max(rescaled if PRIMARY_TEXT_METRIC == "bert_f1_rescaled" else raw, 0.0), 1.0))

## 5. Loading the ground truth and the audit log

`load_trials` reads the audit log and keeps one record per (report, prompt version, repeat): records from earlier pipeline versions that lack a prompt name, records written in test mode (in a study stage) and duplicates (the most recent is kept) are excluded and counted. `load_ground_truth` reads the worksheet that is named after the stage (`prompt_development`, `internal_validation` or `external_validation`), so that no other dataset is opened. `make_demo_truth` generates the ground truth of the demonstration run, which is **not real**.

The column `prompt_id` of the table of trials holds the label of the prompt version (for example `Z4_v2`); the plain name is kept in `prompt_name`. Every table below is grouped by this label. The notebook prints the versions found for each prompt name, and stops if one label contains more than one prompt text.

`apply_exclusions` then removes the reports listed in `EXCLUDED_REPORTS` (Section 1) from the trials and from the ground truth. Report 15 is listed because it was withdrawn from the Coroners Court website, so the model never received it and all 40 of its trials are empty. The function also prints a warning if the prompt versions were not all run on the same reports.

In [5]:
def prompt_label(r: dict) -> str:
    """Return the label of a record's prompt version.

    Args:
        r: A trial record.

    Returns:
        For example 'Z4_v2' for version 2 of prompt Z4; the plain prompt name for records without a version (test report, earlier pipeline versions).
    """
    version = r.get("prompt_version") or 0
    return f"{r['prompt_id']}_v{version}" if version else r["prompt_id"]


def load_trials(path: Path, study_stage: bool) -> pd.DataFrame:
    """Load the trials to score from the audit log.

    Records without a prompt or output (old format) are dropped; in the study stage, test-mode records and records without a prompt version (trials of earlier pipeline versions, which have none) are dropped as well. For duplicates of the same report, prompt version and repeat, the later record is kept. `prompt_id` is replaced by the version label and the plain name is kept in `prompt_name`.

    Args:
        path: The audit log (JSON lines).
        study_stage: Whether this is a study evaluation (drops test-mode records).

    Returns:
        One row per trial, with the added columns prompt_name and prompt_words.

    Raises:
        AssertionError: If one version label covers more than one prompt text.
    """
    recs = [json.loads(l) for l in open(path) if l.strip()]
    keep = [r for r in recs if r.get("prompt_id") and isinstance(r.get("output"), dict) and r["output"]]
    for r in keep:                                                           # from here on `prompt_id` is the label of the version
        r["prompt_name"], r["prompt_id"] = r["prompt_id"], prompt_label(r)
    n_old = len(recs) - len(keep)
    n_test = n_unversioned = 0
    if study_stage:
        n_test = sum(bool(r.get("test_mode")) for r in keep)
        keep = [r for r in keep if not r.get("test_mode")]
        n_unversioned = sum(not r.get("prompt_version") for r in keep)
        keep = [r for r in keep if r.get("prompt_version")]
    latest = {}
    for r in keep:                                                           # later records replace earlier duplicates
        latest[(r["report_no"], r["prompt_id"], r["repeat_index"])] = r
    n_dup = len(keep) - len(latest)
    print(f"{len(recs)} log record(s): {len(latest)} used | {n_old} without prompt/output (old format) | {n_test} test-mode | {n_unversioned} without a prompt version | {n_dup} duplicate(s)")
    df = pd.DataFrame(latest.values())
    if "prompt_hash" not in df:
        df["prompt_hash"] = ""
    df["prompt_hash"] = df["prompt_hash"].fillna("")
    mixed = df[df["prompt_hash"] != ""].groupby("prompt_id")["prompt_hash"].nunique()
    assert (mixed <= 1).all(), f"more than one prompt text under one label: {sorted(mixed[mixed > 1].index)}"
    for name, g in df.groupby("prompt_name", sort=False):
        print(f"  {name}: " + ", ".join(f"{lab} ({n} trials)" for lab, n in g["prompt_id"].value_counts(sort=False).items()))
    system = df["system_text"] if "system_text" in df else df["top_slot_text"]            # v9 name, or the name used by earlier versions
    df["prompt_words"] = system.fillna("").str.split().str.len() + df["user_text"].fillna("").str.split().str.len()
    return df


def load_ground_truth(stage: str) -> dict[int, dict]:
    """Read the ground truth of one stage from the worksheet named after it.

    No other dataset sheet is opened.

    Args:
        stage: The stage, which is also the sheet name.

    Returns:
        A dict mapping each report number to {field: value} for every field in FIELDS.
    """
    gt = pd.read_excel(WORKBOOK, sheet_name=stage, dtype=object)
    return {int(r["Report No."]): {f: r[f] for f in FIELDS} for _, r in gt.iterrows()}


def apply_exclusions(trials: pd.DataFrame, GT: dict) -> tuple[pd.DataFrame, dict]:
    """Remove the reports listed in EXCLUDED_REPORTS from the trials and the ground truth.

    The excluded report is removed completely, so that none of its trials is scored and it does not count as a report that still awaits a trial. Afterwards the function checks that every prompt version was run on the same reports, because the scores of prompts that cover different reports cannot be compared.

    Args:
        trials: The trials, as returned by load_trials.
        GT: The ground truth per report number, as returned by load_ground_truth.

    Returns:
        The trials and the ground truth without the excluded reports.
    """
    for report_no, reason in EXCLUDED_REPORTS.items():
        n_removed = int((trials["report_no"] == report_no).sum())
        print(f"Excluded report {report_no}: {n_removed} trial(s) removed. Reason: {reason}")
    trials = trials[~trials["report_no"].isin(EXCLUDED_REPORTS)].copy()
    GT = {report_no: truth for report_no, truth in GT.items() if report_no not in EXCLUDED_REPORTS}
    covered = trials.groupby("prompt_id")["report_no"].apply(lambda s: tuple(sorted(set(s))))
    if covered.nunique() > 1:
        print("WARNING: the prompt versions were not all run on the same reports; compare only prompts that cover the same reports:")
        for label, reports in covered.items():
            print(f"  {label}: reports {list(reports)}")
    return trials, GT


def make_demo_truth(output: dict) -> dict:
    """Return a fake 'ground truth' made from a model output with deliberate errors.

    DEMONSTRATION ONLY: used to exercise every metric before real ground truth is used.

    Args:
        output: A model output.

    Returns:
        A copy of the output with several fields changed.
    """
    t = json.loads(json.dumps(output))
    t["sex"] = "Female" if t.get("sex") == "Male" else "Male"                # structured field wrong
    t["mental_health_flag"] = True if t.get("mental_health_flag") is not True else False   # flag differs from the model output
    t["custody_served_flag"] = True if t.get("custody_served_flag") is None else None      # blank versus value
    t["place_of_death"] = "Pacific Motorway, Cameron Park"                  # partly different text
    t["manner_of_death"] = ""                                                # ground truth empty, model not
    t["known_health_issues"] = (t.get("known_health_issues") or [])[:-1]    # one list item missing
    t["profession"] = "Student"                                              # different text
    return t

## 6. Scoring every variable of every trial

Each trial yields 28 rows (one per variable) in the long table `scores`. The column `score` is the score used by the best-prompt rule: exact match for flags and structured variables, and the primary text metric (`PRIMARY_TEXT_METRIC`, clipped to the range 0 to 1) for text variables. The other text metrics are stored beside it.

For the six list variables, `score_rows` also computes the **item-level precision, recall and F1** (`item_prf`): each answer is split into its items (case-insensitive), precision is the share of the model's items that are in the ground truth, recall is the share of the ground-truth items that the model gave, and F1 is their harmonic mean. Two empty lists score 1.0 and one empty list 0.0. An item counts only if it matches an item of the ground truth after normalisation of case and white space, so a paraphrased item (for example "type 2 diabetes" for "type 2 diabetes mellitus") is counted as both a missing and an extra item; BERTScore (the primary score) is more tolerant of paraphrase, and the two are read together. Items that contain a comma themselves are split at the comma.

Scoring is done in two steps so that all BERTScore pairs (those against the ground truth and those between repeats) can be sent to BERTScore in a single batch: `score_rows` builds the rows and lists the pairs that need BERTScore, and `finish_scores` adds the BERTScore values once they are available.

In [6]:
TEXT_METRIC_COLUMNS = ["rouge2_f1", "rougeL_f1", "bleu", "bert_f1_raw", "bert_f1_rescaled"]
ITEM_COLUMNS = ["item_precision", "item_recall", "item_f1"]


def item_prf(pred: str, gt: str) -> dict:
    """Return the item-level precision, recall and F1 of a list answer against the ground truth.

    Both answers are split into their items (case-insensitive, duplicates counted once). Precision is the share of the model's items that are in the ground truth, recall is the share of the ground-truth items that the model gave, and F1 is their harmonic mean. Two empty lists score 1.0 on all three, and one empty list scores 0.0.

    Args:
        pred: The model's normalised list answer.
        gt: The ground-truth normalised list answer.

    Returns:
        A dict with item_precision, item_recall and item_f1.
    """
    sp, sg = split_items(pred), split_items(gt)
    if not sp and not sg:
        return dict.fromkeys(ITEM_COLUMNS, 1.0)
    common = len(sp & sg)
    precision = common / len(sp) if sp else 0.0
    recall = common / len(sg) if sg else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return dict(item_precision=precision, item_recall=recall, item_f1=f1)


def score_rows(trials: pd.DataFrame, GT: dict) -> tuple[list[dict], set]:
    """Build the score rows of every variable of every trial, without BERTScore.

    Flags and structured variables are scored by exact match (case-insensitive); text variables get ROUGE and BLEU, with two empty values scoring 1.0, one empty value scoring 0.0 and identical strings scoring 1.0. The pairs that still need BERTScore are returned separately.

    Args:
        trials: The trials, as returned by load_trials.
        GT: The ground truth, as returned by load_ground_truth.

    Returns:
        The rows (one per trial and variable) and the set of (prediction, ground truth) pairs to be sent to BERTScore.
    """
    rows, pairs = [], set()
    for t in trials.itertuples():
        gt_row, out = GT[t.report_no], t.output
        for f in FIELDS:
            ftype = FIELD_TYPE[f]
            if ftype == "flag":
                p, g = to_flag(out.get(f)), to_flag(gt_row.get(f))
            else:
                p, g = to_text(out.get(f), f), to_text(gt_row.get(f), f)
            row = dict(trial_id=t.trial_id, report_no=t.report_no, prompt_id=t.prompt_id, repeat_index=t.repeat_index,
                       field=f, field_type=ftype, pred=p, gt=g, pred_empty=(p == ""), gt_empty=(g == ""))
            row["exact"] = float(p.casefold() == g.casefold())
            if ftype == "text":
                if p == "" or g == "":
                    m = dict.fromkeys(TEXT_METRIC_COLUMNS, float(p == g))
                elif p.casefold() == g.casefold():
                    m = dict.fromkeys(TEXT_METRIC_COLUMNS, 1.0)
                else:
                    m = lexical_metrics(p, g); pairs.add((p, g))
                row.update(m)
            if f in LIST_FIELDS:
                row.update(item_prf(p, g))
            rows.append(row)
    return rows, pairs


def finish_scores(rows: list[dict], bs: dict) -> pd.DataFrame:
    """Add the BERTScore values to the score rows and compute the score column.

    Args:
        rows: The rows from score_rows.
        bs: The BERTScore results, keyed by (prediction, ground truth).

    Returns:
        The long table of scores, with a `score` column.
    """
    for row in rows:
        if row["field_type"] == "text" and (row["pred"], row["gt"]) in bs:
            row["bert_f1_raw"], row["bert_f1_rescaled"] = bs[(row["pred"], row["gt"])]
    df = pd.DataFrame(rows)
    df["score"] = np.where(df["field_type"] == "text", df[PRIMARY_TEXT_METRIC].clip(0, 1), df["exact"])
    return df

## 7. Summary tables

- **by_field**: the mean score of each variable for each prompt (averaged over reports and repeats).
- **by_prompt**: the prompt score (unweighted mean of the 28 variable scores) together with the mean score of each variable group and the prompt length.
- **text_metrics**: for the 14 text variables, the mean of every text metric by prompt. These are reported alongside the primary score so that the conclusions can be checked against ROUGE-2, ROUGE-L and SacreBLEU.
- **flags**: for each flag and prompt, the exact-match accuracy, the share of blank values in the ground truth and in the model output, and the confusion counts (rows: ground truth; columns: model output; `blank` is a valid class). Because most flags are blank for many reports, accuracy alone can be high for a model that returns blanks; the blank rates and the confusion counts show whether that is the case.
- **list_items_by_prompt** and **list_items_by_field**: for the six list variables, the item-level precision, recall and F1 against the ground truth (Section 6). They show whether a list is wrong because items are missing (low recall) or because extra items were added (low precision), which the BERTScore of the joined text does not show. They are reported alongside the primary score and are not part of the best-prompt rule.
- **agreement_by_prompt**, **agreement_by_field** and **agreement_long**: agreement between repeats (Section 4). `agreement_by_prompt` has one row per prompt version, with the percentage of identical answers for each variable group, Krippendorff's alpha (flags, structured variables, and both pooled), the mean pairwise BERTScore F1 of the text variables and the mean Jaccard similarity of the list variables. `agreement_by_field` has one row per prompt version and variable. `agreement_long` has one row per prompt version, report and variable, so that every number can be traced. These tables are empty when no prompt version has two or more repeats of a report.
- **usage**: average token counts per prompt, as recorded in the audit log.
- **diagnostics**: file-number mismatches (the extracted `file_number` differs from the expected one), and the share of model answers that contain placeholder text such as "N/A" or "not stated" instead of an empty value.

In [7]:
FIELD_ORDER = {f: i for i, f in enumerate(FIELDS)}
TECH_ORDER = {"zero_shot": 0, "few_shot": 1, "chain_of_thought": 2}
TEXT_METRICS = ["bert_f1_raw", "bert_f1_rescaled", "rouge2_f1", "rougeL_f1", "bleu"]
PLACEHOLDER = re.compile(r"^(n/?a|none|nil|null|unknown|not (stated|mentioned|specified|provided|applicable|available)|no information.*|-+)$", re.I)


def condition_order(trials: pd.DataFrame) -> list[str]:
    """Return the prompt labels in table order: zero-shot, few-shot, chain-of-thought, then by label.

    Args:
        trials: The trials, as returned by load_trials.

    Returns:
        The ordered list of prompt labels.
    """
    pt = trials.drop_duplicates("prompt_id").assign(_t=lambda d: d["technique"].map(TECH_ORDER).fillna(9))
    return pt.sort_values(["_t", "prompt_id"])["prompt_id"].tolist()


def summarise(trials: pd.DataFrame, scores: pd.DataFrame, order: list[str]) -> dict:
    """Build the accuracy tables (everything except agreement and the ranking).

    Args:
        trials: The trials, as returned by load_trials.
        scores: The long table of scores.
        order: The prompt labels in table order.

    Returns:
        A dict of tables: by_field, by_prompt, text_metrics, text_metrics_by_field, flags, flags_confusion, usage and diagnostics.
    """
    prompt_key = lambda s: s.map({c: i for i, c in enumerate(order)})
    field_key = lambda s: s.map(FIELD_ORDER) if s.name == "field" else prompt_key(s)

    field_cond = scores.groupby(["field_type", "field", "prompt_id"])["score"].mean().unstack("prompt_id")           # per variable and prompt
    by_field = field_cond[order].reset_index().sort_values("field", key=lambda s: s.map(FIELD_ORDER))

    fm = scores.groupby(["prompt_id", "field_type", "field"])["score"].mean().reset_index()                         # per prompt: the best-prompt rule
    cond_score = fm.groupby("prompt_id")["score"].mean().rename("prompt_score")
    by_type = fm.groupby(["prompt_id", "field_type"])["score"].mean().unstack("field_type").add_suffix("_mean")
    meta = trials.groupby("prompt_id").agg(prompt_name=("prompt_name", "first"), prompt_hash=("prompt_hash", "first"),
                                           technique=("technique", "first"), trials=("trial_id", "count"), reports=("report_no", "nunique"),
                                           prompt_words=("prompt_words", "mean"), n_shots=("n_shots", "first"))
    meta["repeats_per_report"] = (meta["trials"] / meta["reports"]).round(2)
    by_prompt = pd.concat([cond_score, by_type, meta], axis=1).reindex(order).reset_index()

    tm = scores[scores["field_type"] == "text"]                                                                      # the 14 text variables
    text_metrics = tm.groupby("prompt_id")[TEXT_METRICS].mean().reindex(order).reset_index()
    text_metrics_by_field = tm.groupby(["field", "prompt_id"])[TEXT_METRICS].mean().reset_index().sort_values(["field", "prompt_id"], key=field_key)

    fl = scores[scores["field_type"] == "flag"].copy()                                                               # flags: accuracy, blanks, confusion
    fl["gt_class"] = fl["gt"].replace("", "blank"); fl["pred_class"] = fl["pred"].replace("", "blank")
    flags_summary = (fl.groupby(["field", "prompt_id"]).agg(n=("score", "size"), accuracy=("score", "mean"),
                     gt_blank_rate=("gt_empty", "mean"), model_blank_rate=("pred_empty", "mean")).reset_index()
                     .sort_values(["field", "prompt_id"], key=field_key))
    flags_confusion = (fl.groupby(["field", "prompt_id", "gt_class", "pred_class"]).size().rename("count").reset_index()
                       .sort_values(["field", "prompt_id", "gt_class", "pred_class"], key=lambda s: s.map(FIELD_ORDER) if s.name == "field" else s))

    usage = (trials.groupby("prompt_id").agg(calls=("trial_id", "count"), mean_input_tokens=("usage_input_tokens", "mean"),
             mean_cached_tokens=("usage_cached_tokens", "mean"), mean_output_tokens=("usage_output_tokens", "mean"),
             mean_reasoning_tokens=("usage_reasoning_tokens", "mean"), mean_total_tokens=("usage_total_tokens", "mean"),
             mean_seconds=("latency_seconds", "mean")).reindex(order).round(4).reset_index())

    li = scores[scores["field"].isin(LIST_FIELDS)]                                                                   # list variables: item-level precision and recall
    list_items_by_field = (li.groupby(["field", "prompt_id"])[ITEM_COLUMNS].mean().reset_index().sort_values(["field", "prompt_id"], key=field_key))
    list_items_by_prompt = li.groupby("prompt_id")[ITEM_COLUMNS].mean().reindex(order).reset_index()

    ph = scores[scores["field_type"] != "flag"].assign(placeholder=lambda d: d["pred"].map(lambda s: bool(PLACEHOLDER.match(s))))
    diagnostics = pd.concat([
        trials.groupby("prompt_id")["file_number_match"].apply(lambda s: int((s == False).sum())).rename("file_number_mismatches"),
        ph.groupby("prompt_id")["placeholder"].mean().rename("placeholder_rate"),
    ], axis=1).reindex(order).reset_index()
    return dict(by_field=by_field, by_prompt=by_prompt, text_metrics=text_metrics, text_metrics_by_field=text_metrics_by_field,
                flags=flags_summary, flags_confusion=flags_confusion, list_items_by_prompt=list_items_by_prompt,
                list_items_by_field=list_items_by_field, usage=usage, diagnostics=diagnostics)

### Agreement tables

`repeat_groups` collects, for every prompt version, report and variable, the answers of the repeats (only where there are at least two). `agreement_pairs` lists the pairs of text answers that need BERTScore. `agreement_tables` then computes the agreement of each group and summarises it by variable (`agreement_by_field`) and by prompt version (`agreement_by_prompt`).

In [8]:
AGREEMENT_PROMPT_COLUMNS = ["prompt_id", "repeats_per_report", "pct_identical_flags", "pct_identical_structured", "pct_identical_text",
                            "alpha_flags", "alpha_structured", "alpha_flags_structured", "items_flags_structured", "items_with_disagreement",
                            "pairwise_bertscore_f1_text", "jaccard_lists", "alpha_note"]
AGREEMENT_FIELD_COLUMNS = ["prompt_id", "field", "field_type", "reports", "repeats_per_report", "pct_identical", "pairwise_bertscore_f1", "jaccard"]
AGREEMENT_LONG_COLUMNS = ["prompt_id", "report_no", "field", "field_type", "repeats", "pct_identical", "pairwise_bertscore_f1", "jaccard"]


def repeat_groups(scores: pd.DataFrame) -> list[dict]:
    """Collect the answers of the repeats for each prompt version, report and variable.

    Args:
        scores: The long table of scores (or the rows from score_rows as a table).

    Returns:
        One dict per group with at least two repeats, holding prompt_id, report_no, field, field_type and the list of normalised answers, in repeat order.
    """
    groups = []
    for (pid, rep, f), g in scores.groupby(["prompt_id", "report_no", "field"], sort=False):
        if len(g) >= 2:
            g = g.sort_values("repeat_index")
            groups.append(dict(prompt_id=pid, report_no=rep, field=f, field_type=g["field_type"].iloc[0], answers=g["pred"].tolist()))
    return groups


def agreement_pairs(groups: list[dict]) -> set:
    """List the pairs of text answers that need BERTScore for the agreement between repeats.

    Args:
        groups: The groups from repeat_groups.

    Returns:
        The set of sorted (answer, answer) pairs: both non-empty and not identical.
    """
    pairs = set()
    for g in groups:
        if g["field_type"] == "text":
            for a, b in combinations(g["answers"], 2):
                if a and b and a.casefold() != b.casefold():
                    pairs.add(tuple(sorted((a, b))))
    return pairs


def agreement_tables(groups: list[dict], bs: dict, order: list[str]) -> dict:
    """Compute the agreement between repeats and summarise it by variable and by prompt.

    Args:
        groups: The groups from repeat_groups.
        bs: The BERTScore results (must contain every pair from agreement_pairs).
        order: The prompt labels in table order.

    Returns:
        A dict with agreement_long, agreement_by_field and agreement_by_prompt. The tables are empty (with their columns) when there are no groups.
    """
    if not groups:
        print("Agreement between repeats: not computed (no prompt version has two or more repeats of a report).")
        return dict(agreement_long=pd.DataFrame(columns=AGREEMENT_LONG_COLUMNS), agreement_by_field=pd.DataFrame(columns=AGREEMENT_FIELD_COLUMNS),
                    agreement_by_prompt=pd.DataFrame(columns=AGREEMENT_PROMPT_COLUMNS))
    rows = []
    for g in groups:
        a = g["answers"]
        row = dict(prompt_id=g["prompt_id"], report_no=g["report_no"], field=g["field"], field_type=g["field_type"], repeats=len(a),
                   pct_identical=pairwise_identical(a), pairwise_bertscore_f1=np.nan, jaccard=np.nan)
        if g["field_type"] == "text":
            row["pairwise_bertscore_f1"] = float(np.mean([pair_similarity(x, y, bs) for x, y in combinations(a, 2)]))
            if g["field"] in LIST_FIELDS:
                row["jaccard"] = float(np.mean([jaccard(x, y) for x, y in combinations(a, 2)]))
        rows.append(row)
    long = pd.DataFrame(rows)[AGREEMENT_LONG_COLUMNS]

    field_key = lambda s: s.map(FIELD_ORDER) if s.name == "field" else s.map({c: i for i, c in enumerate(order)})
    by_field = (long.groupby(["prompt_id", "field", "field_type"], sort=False)
                .agg(reports=("report_no", "nunique"), repeats_per_report=("repeats", "mean"), pct_identical=("pct_identical", "mean"),
                     pairwise_bertscore_f1=("pairwise_bertscore_f1", "mean"), jaccard=("jaccard", "mean")).reset_index()
                .sort_values(["field", "prompt_id"], key=field_key))[AGREEMENT_FIELD_COLUMNS]

    prompt_rows = []
    for pid in order:
        bf = by_field[by_field["prompt_id"] == pid]
        if bf.empty:
            continue
        mine = [g for g in groups if g["prompt_id"] == pid]
        unit = lambda types: [[x.casefold() for x in g["answers"]] for g in mine if g["field_type"] in types]
        flags_u, struct_u = unit({"flag"}), unit({"structured"})
        pooled_u = flags_u + struct_u
        a_flag, a_struct, a_pool = (krippendorff_alpha_nominal(u) for u in (flags_u, struct_u, pooled_u))
        undefined = [n for n, v in (("alpha_flags", a_flag), ("alpha_structured", a_struct), ("alpha_flags_structured", a_pool)) if np.isnan(v)]
        by_type = lambda t, col="pct_identical": bf.loc[bf["field_type"] == t, col].mean()
        prompt_rows.append(dict(
            prompt_id=pid, repeats_per_report=bf["repeats_per_report"].mean(),
            pct_identical_flags=by_type("flag"), pct_identical_structured=by_type("structured"), pct_identical_text=by_type("text"),
            alpha_flags=a_flag, alpha_structured=a_struct, alpha_flags_structured=a_pool,
            items_flags_structured=len(pooled_u), items_with_disagreement=sum(len(set(u)) > 1 for u in pooled_u),
            pairwise_bertscore_f1_text=by_type("text", "pairwise_bertscore_f1"),
            jaccard_lists=bf.loc[bf["field"].isin(LIST_FIELDS), "jaccard"].mean(),
            alpha_note=("undefined (every answer identical, or only one answer value): " + ", ".join(undefined)) if undefined else ""))
    return dict(agreement_long=long, agreement_by_field=by_field, agreement_by_prompt=pd.DataFrame(prompt_rows)[AGREEMENT_PROMPT_COLUMNS])

## 8. Best-prompt rule

The prompt versions are ranked by `prompt_score`, rounded to `TIE_DECIMALS` decimals. A tie is resolved in favour of the shorter prompt (fewer words in the system and user message text), and then of the prompt with fewer worked examples. The ranking is made twice: over all prompts, and within each technique (zero-shot, few-shot, chain-of-thought). **The best prompt of each technique is the candidate for internal validation** (for example `Z4_v2`, `FS2_v1` and `COT1_v1`); after internal validation, the best of those three is the single prompt for external validation. This notebook only reports the rankings. The column `repeats_per_report` shows how many repeats each version has; a version that was tried on fewer repeats than the others is based on less evidence.

`load_prompt_versions` reads the full text of every prompt version from `prompt_versions.jsonl` (written by the prompting pipeline in the results folder), so that the wording of the best prompts can be quoted in the dissertation (Table 3). It returns an empty table for the demonstration run.

In [9]:
PV_COLUMNS = ["prompt_label", "prompt_id", "version", "technique", "n_shots", "prompt_hash", "first_dataset", "first_used", "note", "system_text", "user_text"]


def rank_prompts(by_prompt: pd.DataFrame) -> pd.DataFrame:
    """Rank the prompt versions overall and within each technique.

    Args:
        by_prompt: The table of prompt scores from summarise.

    Returns:
        The ranking: score rounded to TIE_DECIMALS, ties to the shorter prompt and then to fewer examples, with best_of_technique marking the candidates.
    """
    ranking = by_prompt.assign(score_rounded=lambda d: d["prompt_score"].round(TIE_DECIMALS))
    ranking = ranking.sort_values(["score_rounded", "prompt_words", "n_shots"], ascending=[False, True, True]).reset_index(drop=True)
    ranking.insert(0, "rank_overall", ranking.index + 1)
    ranking["rank_in_technique"] = ranking.groupby("technique").cumcount() + 1          # already sorted, so this is the rank within the technique
    ranking["best_of_technique"] = ranking["rank_in_technique"] == 1
    return ranking[["rank_overall", "technique", "rank_in_technique", "prompt_id", "prompt_name", "prompt_score", "flag_mean", "structured_mean", "text_mean",
                    "prompt_words", "n_shots", "trials", "repeats_per_report", "prompt_hash", "best_of_technique"]]


def load_prompt_versions(ranking: pd.DataFrame, study_stage: bool) -> pd.DataFrame:
    """Read the recorded prompt versions of the study (empty for the demonstration run).

    Args:
        ranking: The ranking, used to report versions that are in the log but not in the file.
        study_stage: Whether this is a study evaluation.

    Returns:
        One row per recorded prompt version, with its full text.
    """
    path = RESULTS_ROOT / "prompt_versions.jsonl"
    if not (study_stage and path.exists()):
        return pd.DataFrame(columns=PV_COLUMNS)
    pv = pd.DataFrame([json.loads(l) for l in open(path) if l.strip()])
    pv["prompt_label"] = pv["prompt_id"] + "_v" + pv["version"].astype(str)
    unknown = sorted(set(ranking["prompt_id"]) - set(pv["prompt_label"]))
    if unknown:
        print("Prompt versions in the log but not in prompt_versions.jsonl:", unknown)
    return pv[PV_COLUMNS]

## 9. Export

All tables are written to one workbook, `evaluation_results.xlsx`, in the output folder (`evaluation_results_DEMO.xlsx` for the demonstration run). The `README` sheet records the settings and the versions of the metric libraries, and `scores_long` holds every individual score so that any number in the summaries can be traced. The workbook uses bold header rows, frozen header rows and column widths only; it carries no colours.

In [10]:
EXPORT_SHEETS = ["ranking", "prompt_versions", "by_prompt", "by_field", "text_metrics", "text_metrics_by_field", "flags", "flags_confusion",
                 "list_items_by_prompt", "list_items_by_field", "agreement_by_prompt", "agreement_by_field", "agreement_long", "usage", "diagnostics", "scores_long"]


def _version(pkg: str) -> str:
    """Return the installed version of a package, or 'unknown' if it cannot be found."""
    try:
        return md_.version(pkg)
    except Exception:
        return "unknown"


def export_results(results: dict, out_dir: Path, label: str, demo: bool, log_path: Path) -> Path:
    """Write every results table to one Excel workbook, with a README sheet.

    The workbook is evaluation_results.xlsx, or evaluation_results_DEMO.xlsx for the demonstration run. Formatting is limited to bold header rows, frozen header rows, wrapped headers and column widths (no colours).

    Args:
        results: The results dict from evaluate.
        out_dir: The output folder (created if missing).
        label: A description of what was evaluated, for the README.
        demo: Whether this is the demonstration run.
        log_path: The audit log that was scored, for the README.

    Returns:
        The path of the written workbook.
    """
    trials, ap = results["trials"], results["agreement_by_prompt"]
    tables = {**results, "scores_long": results["scores"]}
    readme = pd.DataFrame({"README": [
        ("DEMONSTRATION ONLY - the ground truth is a fake derived from the model output with deliberate errors. These numbers carry no information about extraction quality." if demo
         else f"Evaluation of {label} against {WORKBOOK}."),
        f"Written {datetime.now():%Y-%m-%d %H:%M}. Audit log: {log_path}. Trials scored: {len(trials)}. Models: {', '.join(sorted(map(str, trials['model'].dropna().unique())))}.",
        "Scoring: flags and structured variables = exact match (blank is a valid class); text variables = BERTScore F1 (primary), with ROUGE-2 F1, ROUGE-L F1 and SacreBLEU sentence BLEU (0 to 1) reported beside it.",
        f"Primary text metric: {PRIMARY_TEXT_METRIC} (clipped to 0..1). BERTScore model {BERT_MODEL} (lang {BERT_LANG}); hashes: {bertscore_batch.hashes}.",
        "List variables also get item-level precision, recall and F1 against the ground truth (exact item match after normalisation; sheets list_items_by_prompt and list_items_by_field); they are reported beside the primary score and are not part of the best-prompt rule.",
        "Empty values (None, '', [], white space) are one value; two empty = 1.0 on every metric, one empty = 0.0; identical normalised strings = 1.0. List variables are joined with ', ' and scored as text.",
        f"Libraries: sacrebleu {_version('sacrebleu')}, rouge-score {_version('rouge-score')}, bert-score {_version('bert-score')}.",
        f"Best-prompt rule: unweighted mean of the 28 variable scores; scores rounded to {TIE_DECIMALS} decimals; ties go to the shorter prompt, then fewer examples; the best prompt of each technique is shortlisted.",
        ("Agreement between repeats (" + (f"{ap['repeats_per_report'].min():g} to {ap['repeats_per_report'].max():g} repeats per report" if len(ap) else "not computed: fewer than two repeats")
         + "): percentage of identical answers for every variable; Krippendorff's alpha (nominal; repeats as raters, report-variable combinations as items) for the flags and the structured variables; text variables = mean pairwise BERTScore F1 between repeats (identical = 1, both empty = 1, one empty = 0), with the percentage of identical answers alongside; list variables also Jaccard similarity of the items."),
        "Prompt versions are scored separately (label = prompt name + _v + version; for example Z4_v2). Sheets: " + ", ".join(EXPORT_SHEETS) + "."]})
    path = out_dir / ("evaluation_results_DEMO.xlsx" if demo else "evaluation_results.xlsx")
    out_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        readme.to_excel(xw, sheet_name="README", index=False)
        for name in EXPORT_SHEETS:
            tables[name].round(4).to_excel(xw, sheet_name=name, index=False)
        for ws in xw.book.worksheets:
            for c in ws[1]:
                c.font = Font(bold=True); c.alignment = Alignment(wrap_text=True, vertical="top")
            ws.freeze_panes = "A2"
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(max(len(str(col[0].value or "")) + 2, 11), 60)
        readme_sheet = xw.book["README"]
        readme_sheet.column_dimensions["A"].width = 170
        readme_sheet["A2"].font = Font(bold=True)                              # the first line states what was evaluated
    print("Written:", path)
    return path

## 10. The evaluation function

`evaluate` is shared by the demonstration run and by the evaluation of a study stage. It scores every variable of every trial, computes the agreement between repeats (when there are two or more repeats of a report), ranks the prompt versions and collects all tables in one dictionary. It does not read any file itself: the trials and the ground truth are passed in, so that the two functions in Sections 11 and 12 decide which data are used.

In [11]:
def evaluate(trials: pd.DataFrame, GT: dict, study_stage: bool) -> dict:
    """Score the trials against the ground truth and measure the agreement between repeats.

    Args:
        trials: The trials, as returned by load_trials.
        GT: The ground truth (or the demonstration ground truth) per report number.
        study_stage: Whether this is a study evaluation (the prompt-version file is read only then).

    Returns:
        A dict with the tables (scores, by_field, by_prompt, text_metrics, text_metrics_by_field, flags, flags_confusion, agreement_long, agreement_by_field, agreement_by_prompt, usage, diagnostics, ranking, prompt_versions) and the inputs (trials, order).
    """
    rows, gt_pairs = score_rows(trials, GT)
    groups = repeat_groups(pd.DataFrame(rows))
    repeat_pairs = agreement_pairs(groups)
    bs = bertscore_batch(sorted(gt_pairs | repeat_pairs))                    # one batch for every pair
    print(f"{len(rows)} variable scores | {len(gt_pairs)} distinct text pairs against the ground truth, {len(repeat_pairs)} between repeats sent to BERTScore")
    scores = finish_scores(rows, bs)
    order = condition_order(trials)
    results = summarise(trials, scores, order)
    results.update(agreement_tables(groups, bs, order))
    results["ranking"] = rank_prompts(results["by_prompt"])
    results["prompt_versions"] = load_prompt_versions(results["ranking"], study_stage)
    results.update(trials=trials, scores=scores, order=order)
    best = results["ranking"][results["ranking"]["best_of_technique"]]
    print("Best prompt of each technique:", dict(zip(best["technique"], best["prompt_id"])))
    return results

## 11. Demonstration run (test report)

`run_demo` scores the records of the test report against a **demonstration ground truth** (`make_demo_truth`, which is not real). It is used to check that the notebook, BERTScore and the agreement metrics work before the real data are scored; it never reads the study ground truth. To test the agreement metrics, run the test report five times in the prompting pipeline first (Section 3 of the prompting pipeline: `REPEATS_PER_DATASET["test_report"] = 5`, with `TEST_MODE = True`); with a single repeat the agreement tables are empty. The outputs go to `evaluation/DEMO/evaluation_results_DEMO.xlsx`.

The numbers of the demonstration run carry no information about extraction quality.

In [12]:
def run_demo(log_paths: list[Path] = DEMO_LOGS, out_dir: Path = EVAL_ROOT / "DEMO") -> dict:
    """Run the whole evaluation on the test report, against a demonstration ground truth.

    Args:
        log_paths: Candidate audit logs of the test report; the first that exists is used.
        out_dir: The output folder.

    Returns:
        The results dict from evaluate, with the path of the workbook under 'workbook_path'.

    Raises:
        FileNotFoundError: If none of the candidate logs exists.
    """
    log_path = next((p for p in log_paths if p.exists()), None)
    if log_path is None:
        raise FileNotFoundError(f"No test-report audit log found in {[str(p) for p in log_paths]}.")
    print(f"DEMONSTRATION RUN | log = {log_path} | outputs -> {out_dir}/")
    trials = load_trials(log_path, study_stage=False)
    GT = {int(r.report_no): make_demo_truth(r.output) for r in trials.drop_duplicates("report_no").itertuples()}
    print("DEMONSTRATION GROUND TRUTH in use - the scores carry no information about extraction quality.")
    results = evaluate(trials, GT, study_stage=False)
    results["workbook_path"] = export_results(results, out_dir, "the demonstration run", demo=True, log_path=log_path)
    return results

Run the demonstration (this cell only; the study stages are run in Section 12).

In [13]:
demo_results = run_demo()
demo_results["agreement_by_prompt"]

DEMONSTRATION RUN | log = results_test/test_report/audit_log.jsonl | outputs -> evaluation/DEMO/
36 log record(s): 15 used | 0 without prompt/output (old format) | 0 test-mode | 0 without a prompt version | 21 duplicate(s)
  Z4: Z4 (5 trials)
  FS2: FS2 (5 trials)
  COT1: COT1 (5 trials)
DEMONSTRATION GROUND TRUTH in use - the scores carry no information about extraction quality.


W1024 23:19:55.443000 90851 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 9531.51it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 28326.12it/s]
[transformers] Rober

420 variable scores | 35 distinct text pairs against the ground truth, 88 between repeats sent to BERTScore
Best prompt of each technique: {'chain_of_thought': 'COT1', 'zero_shot': 'Z4', 'few_shot': 'FS2'}
Written: evaluation/DEMO/evaluation_results_DEMO.xlsx


/opt/anaconda3/envs/eval-env/lib/python3.11/site-packages/bert_score/score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  baselines = torch.from_numpy(


,prompt_id,repeats_per_report,pct_identical_flags,pct_identical_structured,pct_identical_text,alpha_flags,alpha_structured,alpha_flags_structured,items_flags_structured,items_with_disagreement,pairwise_bertscore_f1_text,jaccard_lists,alpha_note
0,Z4,5.0,1.00,0.900000,0.657143,1.000000,0.885827,0.943474,14,1,0.937187,0.927685,
1,FS2,5.0,0.95,0.933333,0.621429,0.892857,0.923483,0.923883,14,2,0.864501,0.777204,
2,COT1,5.0,1.00,1.000000,0.657143,1.000000,1.000000,1.000000,14,0,0.922936,0.938874,


In [14]:
demo_results["agreement_by_field"]

,prompt_id,field,field_type,reports,repeats_per_report,pct_identical,pairwise_bertscore_f1,jaccard
0,Z4,file_number,structured,1,5.0,1.0,NaN,NaN
28,FS2,file_number,structured,1,5.0,1.0,NaN,NaN
56,COT1,file_number,structured,1,5.0,1.0,NaN,NaN
1,Z4,sex,structured,1,5.0,1.0,NaN,NaN
29,FS2,sex,structured,1,5.0,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...
54,FS2,toxicology_results_summary,text,1,5.0,0.1,0.823269,NaN
82,COT1,toxicology_results_summary,text,1,5.0,0.3,0.885147,NaN
27,Z4,coroners_recommendations_summary,text,1,5.0,0.2,0.988792,NaN
55,FS2,coroners_recommendations_summary,text,1,5.0,0.4,0.988792,NaN


## 12. Evaluation of a study stage

`run_stage_evaluation` scores one study stage against its own ground-truth worksheet. It refuses any stage that is not in `ALLOWED_STAGES` (Section 1), and it excludes test-mode records. At this point of the study only `prompt_development` is allowed.

- **Prompt development (5 repeats) and internal validation (5 repeats):** accuracy against the ground truth and agreement between the repeats are both computed.
- **External validation:** the accuracy against the ground truth is the minimum. The external validation set is run once (1 repeat), so the agreement tables are empty; if it is run three times, they are filled in automatically.

The outputs go to `evaluation/<stage>/evaluation_results.xlsx`.

In [15]:
def run_stage_evaluation(stage: str) -> dict:
    """Run the whole evaluation of one study stage against its own ground truth.

    Args:
        stage: 'prompt_development', 'internal_validation' or 'external_validation'.

    Returns:
        The results dict from evaluate, with the path of the workbook under 'workbook_path'.

    Raises:
        PermissionError: If the stage is not in ALLOWED_STAGES.
    """
    if stage not in ALLOWED_STAGES:
        raise PermissionError(f"Stage '{stage}' is not in ALLOWED_STAGES {sorted(ALLOWED_STAGES)}.")
    log_path = RESULTS_ROOT / stage / "audit_log.jsonl"
    print(f"STAGE EVALUATION | stage = {stage} | log = {log_path} | outputs -> {EVAL_ROOT / stage}/")
    trials = load_trials(log_path, study_stage=True)
    GT = load_ground_truth(stage)
    dropped = trials["prompt_id"].isin(EXCLUDED_PROMPT_VERSIONS)
    if dropped.any():
        print("Excluded prompt versions: " + "; ".join(f"{lab} ({n} trials): {EXCLUDED_PROMPT_VERSIONS[lab]}" for lab, n in trials.loc[dropped, "prompt_id"].value_counts().items()))
        trials = trials[~dropped].reset_index(drop=True)
    trials, GT = apply_exclusions(trials, GT)
    missing = sorted(set(GT) - set(trials["report_no"]))
    print(f"{stage}: {len(GT)} reports with ground truth | reports without any trial: {missing or 'none'}")
    assert set(trials["report_no"]) <= set(GT), "the log contains reports outside the evaluated stage"
    results = evaluate(trials, GT, study_stage=True)
    results["workbook_path"] = export_results(results, EVAL_ROOT / stage, f"stage '{stage}'", demo=False, log_path=log_path)
    return results

Run the evaluation of the stage (this cell only).

In [16]:
STAGE = "prompt_development"
results = run_stage_evaluation(STAGE)
results["ranking"]

STAGE EVALUATION | stage = prompt_development | log = results/prompt_development/audit_log.jsonl | outputs -> evaluation/prompt_development/
477 log record(s): 475 used | 0 without prompt/output (old format) | 0 test-mode | 0 without a prompt version | 2 duplicate(s)
  Z0: Z0_v1 (25 trials)
  Z1: Z1_v1 (25 trials)
  Z2: Z2_v1 (25 trials)
  Z3: Z3_v1 (25 trials)
  Z4: Z4_v1 (25 trials)
  FS1: FS1_v1 (25 trials), FS1_v2 (25 trials)
  FS2: FS2_v1 (25 trials), FS2_v2 (25 trials)
  COT1: COT1_v1 (25 trials)
  Z5: Z5_v1 (25 trials)
  Z6: Z6_v1 (25 trials)
  FS1_notes: FS1_notes_v1 (25 trials)
  FS2_notes: FS2_notes_v1 (25 trials)
  FS2_pairs: FS2_pairs_v1 (25 trials)
  COT1_struct: COT1_struct_v1 (25 trials)
  COT2_A: COT2_A_v1 (25 trials)
  COT2_B: COT2_B_v1 (25 trials)
  COT2_C: COT2_C_v1 (25 trials)
Excluded prompt versions: FS1_v1 (25 trials): worked examples contained annotation mismatches (corrected 24 October 2026); superseded by FS1_v2; FS2_v1 (25 trials): worked examples contained a

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 26706.57it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 24562.43it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UN

11900 variable scores | 672 distinct text pairs against the ground truth, 1518 between repeats sent to BERTScore
Best prompt of each technique: {'zero_shot': 'Z6_v1', 'chain_of_thought': 'COT2_B_v1', 'few_shot': 'FS2_pairs_v1'}
Written: evaluation/prompt_development/evaluation_results.xlsx


,rank_overall,technique,rank_in_technique,prompt_id,prompt_name,prompt_score,flag_mean,structured_mean,text_mean,prompt_words,n_shots,trials,repeats_per_report,prompt_hash,best_of_technique
0,1,zero_shot,1,Z6_v1,Z6,0.893490,0.975,0.913333,0.838409,2936.0,0,25,5.0,025c24c499,True
1,2,chain_of_thought,1,COT2_B_v1,COT2_B,0.888814,0.970,0.913333,0.831914,3049.0,0,25,5.0,1021ed699f,True
2,3,chain_of_thought,2,COT2_C_v1,COT2_C,0.888515,0.975,0.913333,0.828459,3012.0,0,25,5.0,b6d96a0782,False
3,4,chain_of_thought,3,COT2_A_v1,COT2_A,0.883121,0.975,0.913333,0.817671,3039.0,0,25,5.0,07c1428592,False
4,5,few_shot,1,FS2_pairs_v1,FS2_pairs,0.848139,0.900,0.900000,0.796278,1561.0,2,25,5.0,640ceeecc3,True
5,6,few_shot,2,FS1_v2,FS1,0.845478,0.900,0.900000,0.790955,1561.0,1,25,5.0,71b99a3168,False
6,7,few_shot,3,FS1_notes_v1,FS1_notes,0.843674,0.900,0.900000,0.787347,1561.0,1,25,5.0,8ce5b7f067,False
7,8,zero_shot,2,Z5_v1,Z5,0.843546,0.900,0.906667,0.784235,2271.0,0,25,5.0,282224c605,False
8,9,few_shot,4,FS2_notes_v1,FS2_notes,0.842760,0.900,0.900000,0.785519,1561.0,2,25,5.0,6506ed7e6a,False
9,10,few_shot,5,FS2_v2,FS2,0.835092,0.900,0.900000,0.770184,1561.0,2,25,5.0,f4593e835d,False


Agreement between repeats, one row per prompt version (Section 4 explains each column).

In [17]:
results["agreement_by_prompt"]

,prompt_id,repeats_per_report,pct_identical_flags,pct_identical_structured,pct_identical_text,alpha_flags,alpha_structured,alpha_flags_structured,items_flags_structured,items_with_disagreement,pairwise_bertscore_f1_text,jaccard_lists,alpha_note
0,Z0_v1,5.0,0.815,0.890000,0.375714,0.697170,0.884349,0.819884,70,21,0.736911,0.588828,
1,Z1_v1,5.0,0.990,0.980000,0.727143,0.975520,0.978827,0.981235,70,2,0.919520,0.789994,
2,Z2_v1,5.0,0.990,0.913333,0.690000,0.975520,0.908468,0.943557,70,6,0.905363,0.779514,
3,Z3_v1,5.0,0.985,0.926667,0.715714,0.964511,0.922521,0.947687,70,6,0.924259,0.852455,
4,Z4_v1,5.0,0.990,0.973333,0.792857,0.973984,0.971775,0.977225,70,2,0.939632,0.894202,
5,Z5_v1,5.0,1.000,0.973333,0.810000,1.000000,0.971775,0.984762,70,2,0.951690,0.899170,
6,Z6_v1,5.0,1.000,0.966667,0.817143,1.000000,0.964725,0.980701,70,2,0.949531,0.883801,
7,FS1_notes_v1,5.0,1.000,0.986667,0.778571,1.000000,0.985882,0.992381,70,1,0.945944,0.870385,
8,FS1_v2,5.0,1.000,0.986667,0.771429,1.000000,0.985882,0.992381,70,1,0.935347,0.873197,
9,FS2_notes_v1,5.0,1.000,0.986667,0.770000,1.000000,0.985882,0.992381,70,1,0.930653,0.857596,


Agreement between repeats, one row per prompt version and variable.

In [18]:
results["agreement_by_field"]

,prompt_id,field,field_type,reports,repeats_per_report,pct_identical,pairwise_bertscore_f1,jaccard
0,Z0_v1,file_number,structured,5,5.0,1.00,NaN,NaN
28,Z1_v1,file_number,structured,5,5.0,1.00,NaN,NaN
56,Z2_v1,file_number,structured,5,5.0,1.00,NaN,NaN
84,Z3_v1,file_number,structured,5,5.0,1.00,NaN,NaN
112,Z4_v1,file_number,structured,5,5.0,1.00,NaN,NaN
...,...,...,...,...,...,...,...,...
391,COT1_struct_v1,coroners_recommendations_summary,text,5,5.0,0.88,0.990438,NaN
167,COT1_v1,coroners_recommendations_summary,text,5,5.0,0.72,0.982622,NaN
419,COT2_A_v1,coroners_recommendations_summary,text,5,5.0,1.00,1.000000,NaN
447,COT2_B_v1,coroners_recommendations_summary,text,5,5.0,0.92,0.998606,NaN


Item-level precision, recall and F1 of the six list variables, one row per prompt version.

In [19]:
results["list_items_by_prompt"]

,prompt_id,item_precision,item_recall,item_f1
0,Z0_v1,0.496318,0.479043,0.482900
1,Z1_v1,0.657579,0.619724,0.632158
2,Z2_v1,0.690412,0.639868,0.656530
3,Z3_v1,0.718533,0.681809,0.692975
4,Z4_v1,0.685054,0.673306,0.676609
5,Z5_v1,0.672785,0.661842,0.664443
6,Z6_v1,0.808745,0.772105,0.786865
7,FS1_notes_v1,0.669129,0.658863,0.659935
8,FS1_v2,0.682709,0.670835,0.674428
9,FS2_notes_v1,0.672579,0.656984,0.660792


Accuracy of every variable, one column per prompt version.

In [20]:
results["by_field"]

prompt_id,field_type,field,Z0_v1,Z1_v1,Z2_v1,Z3_v1,Z4_v1,Z5_v1,Z6_v1,FS1_notes_v1,FS1_v2,FS2_notes_v1,FS2_pairs_v1,FS2_v2,COT1_struct_v1,COT1_v1,COT2_A_v1,COT2_B_v1,COT2_C_v1
12,structured,file_number,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
13,structured,sex,1.000000,1.000000,0.960000,0.960000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.800000,1.000000,1.000000,1.000000,1.000000
8,structured,age_at_death,0.680000,0.920000,0.560000,0.640000,0.880000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.960000,0.960000,1.000000,1.000000,1.000000
11,structured,date_of_findings,0.920000,1.000000,0.840000,0.840000,0.800000,0.840000,0.880000,0.800000,0.800000,0.800000,0.800000,0.800000,0.840000,0.800000,0.880000,0.880000,0.880000
10,structured,date_of_death,0.960000,1.000000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000,0.600000
22,text,place_of_death,0.778278,0.928255,0.934181,0.932354,0.948894,0.948493,0.947290,0.931876,0.947691,0.947691,0.941260,0.948092,0.945213,0.948894,0.936834,0.948894,0.942463
15,text,definitive_cause_of_death,0.712964,0.685759,0.673008,0.686383,0.876880,0.870568,0.868519,0.889429,0.880892,0.879082,0.882818,0.887920,0.866327,0.882818,0.846228,0.847062,0.849051
24,text,potential_causes_of_death,0.096647,0.558503,0.596767,0.797778,0.914187,0.909674,0.902782,0.905377,0.914187,0.905634,0.904986,0.888396,0.912683,0.914187,0.914187,0.916086,0.914187
19,text,manner_of_death,0.439238,0.377695,0.359521,0.373762,0.566400,0.565050,0.558090,0.802309,0.805733,0.746348,0.841976,0.887364,0.628231,0.585700,0.538952,0.600667,0.681818
9,structured,country_of_birth,0.760000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 13. Notes on interpretation

- Text variables are scored with BERTScore as the primary metric because it tolerates paraphrase; ROUGE and BLEU reward exact wording and are reported to show whether the ranking depends on the metric. The variables `toxicology_results_summary` and `coroners_recommendations_summary` are verbatim extracts, so for them the lexical metrics are expected to agree closely with BERTScore.
- Item-level precision and recall of the list variables use exact item matching and are therefore strict about wording; they are most informative when read with BERTScore and the Jaccard agreement. Low precision with high recall means that extra items were added; the reverse means that items were missed.
- `date_of_death` is scored by exact match (not BERTScore), so a date that is wrong by one day scores 0. The match is on the report's own wording, so a correct date written in a different format (for example "8/8/2020" for "8 August 2020") also scores 0; this makes the date-formatting rule in the prompt important.
- Rescaled BERTScore can be negative for unrelated texts; it is clipped at 0 only in the combined score `score` and in the pairwise similarity between repeats, and the unclipped value is retained in `scores_long`.
- Agreement between repeats describes consistency, not accuracy. It is reported beside the accuracy of each prompt and is **not** part of the best-prompt rule.
- Krippendorff's alpha is pooled over several variables. It is undefined when every answer in the pool is identical (see `alpha_note`). For flags that are blank in nearly every report, a high percentage of identical answers can go together with an alpha close to 0, because chance alone would produce the same agreement; the percentage of identical answers and the column `items_with_disagreement` should be read with it. Alpha based on 5 reports is unstable and should be described, not tested.
- The agreement of text variables is the mean pairwise BERTScore F1 between repeats; the percentage of identical answers shows how often the repeats are word-for-word the same. For list variables, the Jaccard similarity shows how much of the content of the lists is shared, regardless of order.
- Comparisons between prompts rest on a small number of reports (5 in prompt development) and should be read as descriptive.